# Tutorial: Direct vs Programmatic Tool Calling — Incident Investigation

Compare both orchestration modes on an adaptive root-cause investigation. The notebook uses deterministic logs, traces, metrics, and deployment records; starts offline; and treats diagnostic quality as a hard gate before cost.


## Audience, prerequisites, and learning goals

This tutorial is for developers who know basic Responses API function calling and want to understand when semantic branching changes the Direct-versus-Programmatic tradeoff.

Prerequisites:

- Python 3.11 or newer and `uv`
- An OpenAI API key only for optional live cells
- Familiarity with function-call and function-call-output items

You will learn to keep an adaptive investigation fair, preserve program caller linkage across multiple pauses, inspect the selected evidence route, and compare measured tokens and estimated cost only when the diagnosis is correct and grounded.


## Outline

1. Configure safe execution controls.
2. Inspect three deterministic incidents and their evidence graph.
3. Compare identical function schemas and arm-specific orchestration.
4. Validate evidence availability offline.
5. Optionally run one case, inspect its route, and then run all cases.
6. Interpret results and complete an adaptive-routing exercise.


## 1. Setup

The import cell locates the project package. It neither loads nor prints an API key.


In [1]:
from __future__ import annotations

import json
import os
import sys
import uuid
from pathlib import Path

from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "ptc_benchmark").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "ptc_benchmark").exists():
    raise RuntimeError("Run this notebook from the project root or notebooks directory.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ptc_benchmark.config import configured_model, load_local_environment, require_api_key
from ptc_benchmark.incident import INCIDENT_CASE_IDS, build_incident_scenario, collect_evidence_ids
from ptc_benchmark.incident_evaluation import evaluate_incident_run
from ptc_benchmark.pricing import estimate_run_cost, load_pricing_catalog
from ptc_benchmark.reporting import append_jsonl, comparison_rows, markdown_table, request_timeline
from ptc_benchmark.runner import RunConfig, ToolCallingRunner


### Safe execution controls

Live execution and the six-run case suite are disabled independently. The single walkthrough defaults to the database-pool case because its route uses all four function types.


In [2]:
RUN_LIVE = False # Set to True to enable live API calls and associated cost.
RUN_ALL_CASES = False # Set to True for all cases; requires RUN_LIVE = True.

CASE_ID = "database-pool-exhaustion"
MODEL = configured_model("gpt-5.6")
REASONING_EFFORT = "medium"
MAX_REQUESTS = 16  # Allows adaptive/PTC continuation turns while keeping cost bounded.

PRICING_PATH = Path(
    os.getenv(
        "OPENAI_PRICING_PATH",
        PROJECT_ROOT / "pricing" / "openai_pricing_2026-08-14.json",
    )
)
RESULTS_PATH = PROJECT_ROOT / "results" / "incident_runs.jsonl"

print({
    "RUN_LIVE": RUN_LIVE,
    "RUN_ALL_CASES": RUN_ALL_CASES,
    "case_id": CASE_ID,
    "model": MODEL,
    "reasoning_effort": REASONING_EFFORT,
    "max_requests": MAX_REQUESTS,
})


{'RUN_LIVE': True, 'RUN_ALL_CASES': True, 'case_id': 'database-pool-exhaustion', 'model': 'gpt-5.6', 'reasoning_effort': 'medium', 'max_requests': 16}


## 2. Why this task is different

Inventory replenishment has a fixed fan-out and arithmetic reduction. Incident investigation is adaptive: the meaning of the initial log and metric results determines the next trace, service metric, or deployment lookup.

Both arms receive the same model, reasoning effort, fixtures, output contract, and four function schemas. Both must start with exactly two independent entry-service observations. Direct may parallelize calls once evidence justifies them; Programmatic may retain intermediate evidence and branch inside generated JavaScript. Each run has its own cache key, so neither arm warms the other.


## 3. Deterministic incidents and answer oracle

The three incidents cover database pool exhaustion, an expired TLS certificate, and a response-schema mismatch. Fixtures contain distracting but non-causal events. The local oracle is used only for scoring; the model must retrieve every evidence ID it cites.


In [3]:
case_rows = []
for case_id in INCIDENT_CASE_IDS:
    case = build_incident_scenario(case_id)
    expected = case.expected_result()
    case_rows.append({
        "case_id": case_id,
        "affected_service": expected["affected_service"],
        "root_cause": expected["root_cause"],
        "required_evidence": len(expected["evidence_ids"]),
    })
display(Markdown(markdown_table(case_rows)))


case_id,affected_service,root_cause,required_evidence
database-pool-exhaustion,payment-api,database_connection_pool_exhaustion,4
expired-tls-certificate,identity-api,expired_tls_certificate,3
pricing-schema-mismatch,pricing-api,pricing_schema_version_mismatch,4


In [4]:
scenario = build_incident_scenario(CASE_ID)
display(Markdown("### Oracle result\n```json\n" + json.dumps(scenario.expected_result(), indent=2) + "\n```"))
display(Markdown("### Entry-service logs\n```json\n" + json.dumps(scenario.logs[scenario.entry_service], indent=2) + "\n```"))


Oracle result ¶ { 
 "incident_id" : "database-pool-exhaustion" , 
 "root_cause" : "database_connection_pool_exhaustion" , 
 "affected_service" : "payment-api" , 
 "confidence" : "high" , 
 "evidence_ids" : [ 
 "deploy-payment-pool-capacity" , 
 "log-checkout-payment-timeout" , 
 "metric-payment-db-pool-wait" , 
 "span-payment-db-pool" 
 ], 
 "recommended_action" : "restore_database_pool_capacity" 
 }

Entry-service logs ¶ [ 
 { 
 "evidence_id" : "log-checkout-payment-timeout" , 
 "timestamp" : "2026-08-06T10:02:12Z" , 
 "severity" : "ERROR" , 
 "error_code" : "UPSTREAM_TIMEOUT" , 
 "message" : "Checkout failed after payment-api timed out while authorizing the order." , 
 "trace_id" : "trace-payment-001" , 
 "dependency" : "payment-api" 
 }, 
 { 
 "evidence_id" : "log-checkout-cache-noise" , 
 "timestamp" : "2026-08-06T10:03:44Z" , 
 "severity" : "WARN" , 
 "error_code" : "CACHE_MISS" , 
 "message" : "Promotion cache miss recovered on retry and did not fail the request." , 
 "trace_id" : "trace-noise-001" , 
 "dependency" : "promotion-cache" 
 }, 
 { 
 "evidence_id" : "log-checkout-client-noise" , 
 "timestamp" : "2026-08-06T10:05:09Z" , 
 "severity" : "WARN" , 
 "error_code" : "CLIENT_CANCELLED" , 
 "message" : "Client disconnected after 120 ms." , 
 "trace_id" : "trace-noise-002" , 
 "dependency" : null 
 } 
 ]

## 4. Equivalent tool surfaces

`search_logs`, `get_trace`, `get_service_metrics`, and `list_recent_deployments` have strict input and output schemas. The function definitions differ only in `allowed_callers`; Programmatic additionally receives the hosted runtime tool.


In [5]:
tool_rows = []
for arm in ("direct", "programmatic"):
    for tool in scenario.tool_definitions(arm):
        tool_rows.append({
            "arm": arm,
            "type": tool["type"],
            "name": tool.get("name", "hosted runtime"),
            "allowed_callers": tool.get("allowed_callers", []),
            "strict_output": "output_schema" in tool,
        })
display(Markdown(markdown_table(tool_rows)))


arm,type,name,allowed_callers,strict_output
direct,function,search_logs,['direct'],True
direct,function,get_trace,['direct'],True
direct,function,list_recent_deployments,['direct'],True
direct,function,get_service_metrics,['direct'],True
programmatic,function,search_logs,['programmatic'],True
programmatic,function,get_trace,['programmatic'],True
programmatic,function,list_recent_deployments,['programmatic'],True
programmatic,function,get_service_metrics,['programmatic'],True
programmatic,programmatic_tool_calling,hosted runtime,[],False


## 5. Shared contract, different orchestration

The common contract fixes the two-call start, diagnosis taxonomy, stopping rule, JSON shape, and evidence requirements. Direct is told to inspect each result before selecting the next calls. Programmatic is told to branch inside JavaScript and emit only the reduced result.


In [6]:
for arm in ("direct", "programmatic"):
    instructions, user_prompt = scenario.prompt(arm)
    orchestration = instructions.split("<tool_orchestration>", 1)[1].split("</tool_orchestration>", 1)[0].strip()
    display(Markdown(f"### {arm.title()} orchestration\n```text\n{orchestration}\n```"))
print(f"Shared user prompt: {user_prompt}")


Direct orchestration ¶ Use Direct Tool Calling. Make only the two required starting calls initially. Inspect
their semantic content before selecting subsequent calls. Parallelize independent
calls only after the preceding evidence justifies them. Do not use a generated program.

Programmatic orchestration ¶ Use Programmatic Tool Calling for the investigation. In generated JavaScript, make
the two required starting calls with Promise.all, inspect their structured fields and
natural-language messages, and branch to later trace, metric, or deployment calls.
Keep intermediate results inside the program. Emit exactly the required result with
text(JSON.stringify(result)). Do not call investigation functions directly.

Shared user prompt: Find the root cause of incident database-pool-exhaustion, cite the evidence, and recommend the immediate remediation.


## 6. Offline quality checks

Before making an API request, confirm that each oracle evidence ID exists in its case and every function exposes a strict output schema. Live scoring additionally checks the exact diagnosis, final JSON, explanation completeness, retrieved-evidence grounding, required start, absence of duplicate calls, and caller linkage.


In [7]:
offline_rows = []
for case_id in INCIDENT_CASE_IDS:
    case = build_incident_scenario(case_id)
    available = set()
    for collection in (case.logs, case.traces, case.deployments, case.metrics):
        available.update(collect_evidence_ids(collection))
    required = set(case.expected_result()["evidence_ids"])
    offline_rows.append({
        "case_id": case_id,
        "oracle_evidence_available": required.issubset(available),
        "strict_function_outputs": all(
            "output_schema" in tool
            for tool in case.tool_definitions("direct")
            if tool["type"] == "function"
        ),
    })
assert all(row["oracle_evidence_available"] and row["strict_function_outputs"] for row in offline_rows)
display(Markdown(markdown_table(offline_rows)))


case_id,oracle_evidence_available,strict_function_outputs
database-pool-exhaustion,True,True
expired-tls-certificate,True,True
pricing-schema-mismatch,True,True


## 7. Optional single-case live comparison

This cell does nothing unless `RUN_LIVE = True`. It runs the selected case once per arm, evaluates quality, calculates estimated model cost from the dated snapshot, and appends the trace to a gitignored JSONL file.


In [8]:
live_results = {}

if not RUN_LIVE:
    print("Live comparison skipped. Set RUN_LIVE = True to opt in to API usage and cost.")
else:
    from openai import OpenAI

    load_local_environment(PROJECT_ROOT)
    require_api_key()
    pricing = load_pricing_catalog(PRICING_PATH)
    runner = ToolCallingRunner(OpenAI())
    run_config = RunConfig(
        model=MODEL,
        reasoning_effort=REASONING_EFFORT,
        max_requests=MAX_REQUESTS,
    )
    comparison_id = f"incident-{CASE_ID}-{uuid.uuid4().hex[:8]}"

    for arm in ("direct", "programmatic"):
        run = runner.run(arm=arm, scenario=scenario, config=run_config, run_id=comparison_id)
        evaluation = evaluate_incident_run(run, scenario)
        cost = estimate_run_cost(run, pricing)
        live_results[arm] = (run, evaluation, cost)
        append_jsonl(RESULTS_PATH, run, evaluation, cost)

    display(Markdown(markdown_table(comparison_rows(live_results.values()))))
    for arm, (_, evaluation, _) in live_results.items():
        if not evaluation.passed:
            print(f"{arm} failed quality gates: {evaluation.failures}")


arm,passed,requests,tool_calls,input_tokens,cached_tokens,cache_write_tokens,output_tokens,reasoning_tokens,estimated_cost_usd,end_to_end_seconds
direct,False,4,5,5149,2602,1823,818,334,0.040855,23.429
programmatic,False,6,6,4968,1475,3356,2116,535,0.085877,37.886


direct failed quality gates: ('The structured execution result does not match the incident oracle.', 'RESULT_JSON does not match the incident oracle.')
programmatic failed quality gates: ('The structured execution result does not match the incident oracle.', 'RESULT_JSON does not match the incident oracle.', 'EXPLANATION omits the diagnosis, action, service, or required evidence IDs.')


## 8. Inspect the adaptive route

The route table shows which evidence caused subsequent calls. Request 1 should contain only the two required entry observations. Later requests should follow the trace and dependency discovered there, without repeating an identical call.


In [9]:
def route_rows(run):
    rows = []
    for call in run.tool_calls:
        rows.append({
            "request": call.request_index + 1,
            "tool": call.name,
            "arguments": json.dumps(call.arguments, sort_keys=True),
            "evidence_ids": ", ".join(sorted(collect_evidence_ids(call.output))),
            "caller": "program" if call.caller else "direct",
        })
    return rows


if not live_results:
    print("No live routes to display.")
else:
    for arm, (run, evaluation, cost) in live_results.items():
        display(Markdown(f"### {arm.title()} request timeline"))
        display(Markdown(markdown_table(request_timeline(run))))
        display(Markdown(f"### {arm.title()} evidence route"))
        display(Markdown(markdown_table(route_rows(run))))
        display(Markdown(f"**Quality:** `{evaluation.passed}`  \n**Estimated cost:** `${cost.total_cost:.6f}`  \n**End-to-end latency:** `{run.total_latency_seconds:.3f}s`"))


Direct request timeline ¶

request,output_types,input_tokens,cached_tokens,cache_write_tokens,output_tokens,latency_seconds
1,"reasoning, function_call, function_call",715,0,0,151,7.07
2,"reasoning, function_call",1227,0,1224,33,2.111
3,"reasoning, function_call, function_call",1381,1224,154,226,6.056
4,"reasoning, message",1826,1378,445,408,8.19


Direct evidence route ¶

request,tool,arguments,evidence_ids,caller
1,search_logs,"{""query"": ""errors timeouts failures"", ""service"": ""checkout-api"", ""window_end"": ""2026-08-06T10:20:00Z"", ""window_start"": ""2026-08-06T10:00:00Z""}","log-checkout-cache-noise, log-checkout-client-noise, log-checkout-payment-timeout",direct
1,get_service_metrics,"{""metric"": ""error_rate"", ""service"": ""checkout-api"", ""window_end"": ""2026-08-06T10:20:00Z"", ""window_start"": ""2026-08-06T10:00:00Z""}",metric-checkout-error-spike,direct
2,get_trace,"{""trace_id"": ""trace-payment-001""}","span-checkout-payment, span-payment-db-pool",direct
3,get_service_metrics,"{""metric"": ""db_pool_wait_ms"", ""service"": ""payment-api"", ""window_end"": ""2026-08-06T10:20:00Z"", ""window_start"": ""2026-08-06T10:00:00Z""}",metric-payment-db-pool-wait,direct
3,list_recent_deployments,"{""service"": ""payment-api"", ""window_end"": ""2026-08-06T10:20:00Z"", ""window_start"": ""2026-08-06T10:00:00Z""}",deploy-payment-pool-capacity,direct


Quality: False 
 Estimated cost: $0.040855 
 End-to-end latency: 23.429s

Programmatic request timeline ¶

request,output_types,input_tokens,cached_tokens,cache_write_tokens,output_tokens,latency_seconds
1,"reasoning, program, function_call, function_call",1478,0,1344,1922,29.998
2,function_call,0,0,0,0,0.82
3,function_call,0,0,0,0,1.083
4,function_call,0,0,0,0,1.24
5,function_call,0,0,0,0,0.984
6,"program_output, reasoning, message",3490,1475,2012,194,3.759


Programmatic evidence route ¶

request,tool,arguments,evidence_ids,caller
1,search_logs,"{""query"": ""errors timeouts failures"", ""service"": ""checkout-api"", ""window_end"": ""2026-08-06T10:20:00Z"", ""window_start"": ""2026-08-06T10:00:00Z""}","log-checkout-cache-noise, log-checkout-client-noise, log-checkout-payment-timeout",program
1,get_service_metrics,"{""metric"": ""error_rate"", ""service"": ""checkout-api"", ""window_end"": ""2026-08-06T10:20:00Z"", ""window_start"": ""2026-08-06T10:00:00Z""}",metric-checkout-error-spike,program
2,get_trace,"{""trace_id"": ""trace-payment-001""}","span-checkout-payment, span-payment-db-pool",program
3,search_logs,"{""query"": ""connection pool exhausted timeout database"", ""service"": ""payment-api"", ""window_end"": ""2026-08-06T10:20:00Z"", ""window_start"": ""2026-08-06T10:00:00Z""}",log-payment-pool-wait,program
4,get_service_metrics,"{""metric"": ""db_pool_wait_ms"", ""service"": ""payment-api"", ""window_end"": ""2026-08-06T10:20:00Z"", ""window_start"": ""2026-08-06T10:00:00Z""}",metric-payment-db-pool-wait,program
5,list_recent_deployments,"{""service"": ""payment-api"", ""window_end"": ""2026-08-06T10:20:00Z"", ""window_start"": ""2026-08-06T10:00:00Z""}",deploy-payment-pool-capacity,program


Quality: False 
 Estimated cost: $0.085877 
 End-to-end latency: 37.886s

In [10]:
if live_results:
    programmatic_run = live_results["programmatic"][0]
    if programmatic_run.generated_programs:
        display(Markdown("### Generated JavaScript\n```javascript\n" + programmatic_run.generated_programs[-1] + "\n```"))
    if programmatic_run.program_outputs:
        display(Markdown("### Program output\n```json\n" + json.dumps(programmatic_run.program_outputs[-1], indent=2) + "\n```"))
else:
    print("Run the optional live comparison to inspect generated code and program output.")


Generated JavaScript ¶ const window_start = "2026-08-06T10:00:00Z" ; 
 const window_end = "2026-08-06T10:20:00Z" ; 

 const [ checkoutLogs , checkoutErrorRate ] = await Promise . all ([ 
 tools . search_logs ({ 
 service : "checkout-api" , 
 window_start , 
 window_end , 
 query : "errors timeouts failures" 
 }), 
 tools . get_service_metrics ({ 
 service : "checkout-api" , 
 window_start , 
 window_end , 
 metric : "error_rate" 
 }) 
 ]); 

 const events = checkoutLogs . events || []; 
 const poolPattern = /pool|connection|database|db|checkout|timeout/i ; 
 const tlsPattern = /tls|certificate|handshake|x509|expired/i ; 
 const schemaPattern = /schema|parse|deserialize|response version|unknown field/i ; 

 let signal = "pool" ; 
 const combinedInitial = [ 
 ... events . map ( e => ` ${ e . error_code } ${ e . message } ${ e . dependency || "" } ` ), 
 checkoutErrorRate . summary || "" 
 ]. join ( " " ); 
 if ( tlsPattern . test ( combinedInitial ) && ! poolPattern . test ( combinedInitial )) signal = "tls" ; 
 if ( schemaPattern . test ( combinedInitial ) && ! poolPattern . test ( combinedInitial )) signal = "schema" ; 

 const leadEvent = 
 events . find ( e => signal === "pool" && poolPattern . test ( ` ${ e . error_code } ${ e . message } ` )) || 
 events . find ( e => signal === "tls" && tlsPattern . test ( ` ${ e . error_code } ${ e . message } ` )) || 
 events . find ( e => signal === "schema" && schemaPattern . test ( ` ${ e . error_code } ${ e . message } ` )) || 
 events . find ( e => e . trace_id ) || 
 events [ 0 ]; 

 let trace = { found : false , spans : [] }; 
 if ( leadEvent && leadEvent . trace_id ) { 
 trace = await tools . get_trace ({ trace_id : leadEvent . trace_id }); 
 } 

 const failedSpans = ( trace . spans || []). filter ( s => s . status !== "OK" ); 
 const causalSpan = 
 failedSpans . find ( s => signal === "pool" && poolPattern . test ( ` ${ s . operation } ${ s . message } ` )) || 
 failedSpans . find ( s => signal === "tls" && tlsPattern . test ( ` ${ s . operation } ${ s . message } ` )) || 
 failedSpans . find ( s => signal === "schema" && schemaPattern . test ( ` ${ s . operation } ${ s . message } ` )) || 
 failedSpans [ failedSpans . length - 1 ] || 
 ( trace . spans || [])[ 0 ]; 

 let affectedService = 
 ( causalSpan && causalSpan . service ) || 
 ( leadEvent && leadEvent . dependency ) || 
 "checkout-api" ; 

 if ( affectedService === "checkout-api" && leadEvent && leadEvent . dependency ) { 
 affectedService = leadEvent . dependency ; 
 } 

 let downstreamLogs = { events : [] }; 
 if ( affectedService !== "checkout-api" ) { 
 downstreamLogs = await tools . search_logs ({ 
 service : affectedService , 
 window_start , 
 window_end , 
 query : 
 signal === "pool" ? "connection pool exhausted timeout database" : 
 signal === "tls" ? "TLS handshake certificate expired" : 
 "response parse schema version" 
 }); 
 } 

 const allDiagnosticText = [ 
 combinedInitial , 
 ...( trace . spans || []). map ( s => ` ${ s . service } ${ s . operation } ${ s . message } ` ), 
 ...( downstreamLogs . events || []). map ( e => ` ${ e . error_code } ${ e . message } ${ e . dependency || "" } ` ) 
 ]. join ( " " ); 

 if ( tlsPattern . test ( allDiagnosticText ) && ! /pool exhausted|pool saturation|connection pool/i . test ( allDiagnosticText )) signal = "tls" ; 
 else if ( schemaPattern . test ( allDiagnosticText ) && ! /pool exhausted|pool saturation|connection pool/i . test ( allDiagnosticText )) signal = "schema" ; 
 else if ( poolPattern . test ( allDiagnosticText )) signal = "pool" ; 

 const metricName = 
 signal === "pool" ? "db_pool_wait_ms" : 
 signal === "tls" ? "tls_handshake_errors" : 
 "response_parse_errors" ; 

 const causalMetric = await tools . get_service_metrics ({ 
 service : affectedService , 
 window_start , 
 window_end , 
 metric : metricName 
 }); 

 let deployment = { deployments : [] }; 
 if ( signal === "pool" || signal === "schema" ) { 
 deployment = await tools . list_recent_depl

Program output ¶ { 
 "incident_id" : "database-pool-exhaustion" , 
 "root_cause" : "database_connection_pool_exhaustion" , 
 "affected_service" : "payment-api" , 
 "confidence" : "high" , 
 "evidence_ids" : [ 
 "deploy-payment-pool-capacity" , 
 "log-payment-pool-wait" , 
 "metric-checkout-error-spike" , 
 "metric-payment-db-pool-wait" , 
 "span-payment-db-pool" 
 ], 
 "recommended_action" : "restore_database_pool_capacity" 
 }

## 9. Optional three-case suite

One walkthrough explains the mechanics; the full suite checks whether the tradeoff changes across three semantic branches. Enabling this section performs six live runs. Each case pair uses a new isolated cache key.


In [11]:
suite_rows = []

if not RUN_ALL_CASES:
    print("All-case suite skipped. This is the safe default.")
elif not RUN_LIVE:
    raise RuntimeError("RUN_ALL_CASES requires RUN_LIVE = True.")
else:
    from openai import OpenAI

    load_local_environment(PROJECT_ROOT)
    require_api_key()
    pricing = load_pricing_catalog(PRICING_PATH)
    runner = ToolCallingRunner(OpenAI())
    run_config = RunConfig(
        model=MODEL,
        reasoning_effort=REASONING_EFFORT,
        max_requests=MAX_REQUESTS,
    )

    for case_id in INCIDENT_CASE_IDS:
        case = build_incident_scenario(case_id)
        pair_id = f"incident-suite-{case_id}-{uuid.uuid4().hex[:8]}"
        for arm in ("direct", "programmatic"):
            run = runner.run(arm=arm, scenario=case, config=run_config, run_id=pair_id)
            evaluation = evaluate_incident_run(run, case)
            cost = estimate_run_cost(run, pricing)
            append_jsonl(RESULTS_PATH, run, evaluation, cost)
            row = comparison_rows([(run, evaluation, cost)])[0]
            row = {"case_id": case_id, **row}
            suite_rows.append(row)

    display(Markdown(markdown_table(suite_rows)))


case_id,arm,passed,requests,tool_calls,input_tokens,cached_tokens,cache_write_tokens,output_tokens,reasoning_tokens,estimated_cost_usd,end_to_end_seconds
database-pool-exhaustion,direct,False,5,6,7120,4406,1987,825,291,0.043007,22.747
database-pool-exhaustion,programmatic,False,5,5,4005,1475,2393,1182,229,0.051839,20.733
expired-tls-certificate,direct,False,5,4,5935,3598,1611,708,344,0.036738,14.072
expired-tls-certificate,programmatic,False,4,4,4417,1474,2806,1626,63,0.067739,38.865
pricing-schema-mismatch,direct,False,4,4,4750,2472,1560,685,278,0.035126,18.94
pricing-schema-mismatch,programmatic,False,3,3,4532,1469,2926,1716,257,0.071187,27.849


## 10. Interpretation

Read results in this order:

1. Exclude runs that fail any diagnostic quality gate.
2. Compare the routes: unnecessary calls, repeats, and number of model requests.
3. Compare input, cached-input, cache-write, output, and reasoning tokens across the complete loop.
4. Compare request latency and task end-to-end latency separately.
5. Apply the same dated pricing snapshot and label the result as an estimate.

Programmatic processing can keep intermediate evidence out of later model context, but adaptive semantic judgment may be easier for Direct Tool Calling. Generated code and extra reasoning also consume tokens. This benchmark is designed to measure that tension, not assume a winner.


## Exercise

Before running the TLS case, predict the minimal justified route after the two starting calls. Then compare your route with the model's actual calls. A good answer names the trace lookup and the one service-level metric needed to corroborate the certificate evidence; it does not add unrelated deployment or database checks.


In [12]:
def quality_adjusted_delta(rows, metric):
    by_arm = {row["arm"]: row for row in rows}
    if set(by_arm) != {"direct", "programmatic"}:
        raise ValueError("Expected one direct and one programmatic row")
    if not all(by_arm[arm]["passed"] for arm in by_arm):
        return None
    return by_arm["programmatic"][metric] - by_arm["direct"][metric]


if live_results:
    rows = comparison_rows(live_results.values())
    print("Input-token delta:", quality_adjusted_delta(rows, "input_tokens"))
    print("Tool-call delta:", quality_adjusted_delta(rows, "tool_calls"))
    print("Estimated-cost delta:", quality_adjusted_delta(rows, "estimated_cost_usd"))
else:
    print("Run the optional live comparison to calculate quality-adjusted deltas.")


Input-token delta: None
Tool-call delta: None
Estimated-cost delta: None


## Pitfalls and extensions

- Do not let either arm inspect the oracle during a live run.
- Do not force Direct into a serial baseline; independent calls may be parallel.
- Do not drop `caller` from program-owned function outputs when a program pauses more than once.
- Do not reward exhaustive tool use; unnecessary calls weaken both efficiency and investigation quality.
- Do not cite evidence IDs that were never retrieved.
- Do not call a failed but cheaper diagnosis a winner.

Useful extensions include noisy multi-tenant logs, fixed local tool latency, partial traces, alternative stopping thresholds, and repeated live runs for pass-rate confidence intervals.
